# Custom CNN: unattended parameter search

Run **after** `colab_smoke_test.ipynb` passed. Use *Run All* and leave the PC awake (a sleeping laptop disconnects the Colab session).

| Stage | What | Budget |
|---|---|---|
| A | one-factor ablations around the baseline, 20 epochs each | `STAGE_A_END_H` |
| B | Optuna study (architecture + optimiser knobs), 15 epochs per trial, bad trials pruned | until `STAGE_B_END_H` |
| C | the best trials trained longer on folds 0 and 1, with test predictions | until `TOTAL_HOURS` |

Everything is exported to Google Drive (`dl_lab2/results/<run>/`) after every run. If the session dies, run the notebook again: finished runs are skipped
and the Optuna study resumes from its backup.

In [ ]:
TOTAL_HOURS = 3.5
STAGE_A_END_H = 0.8   # no new ablation starts after this many hours
STAGE_B_END_H = 2.4   # Optuna starts no new trial after this
A_EPOCHS, B_EPOCHS, FINAL_EPOCHS = 20, 15, 40
N_TRIALS = 60         # upper bound; the time limit usually stops the study first

REPO_URL = "https://github.com/vumenira/CNN_for_multilabel_classification.git"
BRANCH = "param-tuning"

In [ ]:
import os, subprocess, sys, time, traceback
from pathlib import Path

T0 = time.time()
REPO, ENV_FILE = Path("/content/CNN_for_multilabel_classification"), Path("/content/.env")

if not REPO.exists():
    url, token = REPO_URL, None
    if ENV_FILE.exists():
        lines = ENV_FILE.read_text().splitlines()
        token = next((l.split("=", 1)[1].strip() for l in lines if l.startswith("GITHUB_TOKEN=")), None)
        if token:
            url = url.replace("https://", f"https://{token}@")
    r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, str(REPO)], capture_output=True, text=True)
    assert r.returncode == 0, r.stderr.replace(token, "***") if token else r.stderr
sys.path.insert(0, str(REPO))
os.chdir(REPO)

packages = ["lightning", "timm", "mlflow", "albumentations>=2.0.8", "imagehash",
            "iterative-stratification", "torchmetrics", "optuna", "kaggle", "tqdm"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

from src import colab, results, sweep
from src.data import build_cache

assert colab.mount_drive(), "Drive is not available: stop here, otherwise results are lost with the session"
RESULTS = colab.DRIVE_DIR / "results"
RESULTS.mkdir(exist_ok=True)
colab.load_env(ENV_FILE)
colab.ensure_dataset(colab.DRIVE_DIR)
build_cache("train", 224)
build_cache("test", 224)
print(f"setup took {(time.time() - T0) / 60:.1f} min; results go to {RESULTS}")

## Stage A: one-factor ablations (baseline architecture, 20 epochs)

In [ ]:
def ablation(name, **changes):
    return dict(name=f"A-{name}", stage="A", epochs=A_EPOCHS, **changes)


ABLATIONS = [
    ablation("base"),
    ablation("posweight-sqrt", pos_weight="sqrt"),
    ablation("lr-3e-4", lr=3e-4),
    ablation("lr-3e-3", lr=3e-3),
    ablation("wd-1e-2", weight_decay=1e-2),
    ablation("no-batchnorm", use_batchnorm=False),
]

try:
    sweep.run_list(ABLATIONS, RESULTS, deadline=T0 + STAGE_A_END_H * 3600)
except Exception:
    traceback.print_exc()

## Stage B: Optuna (maximises validation macro AP)

In [ ]:
study = None
try:
    study = sweep.run_optuna(
        "B", n_trials=N_TRIALS, results_dir=RESULTS,
        overrides=dict(epochs=B_EPOCHS),
        timeout=max(60, T0 + STAGE_B_END_H * 3600 - time.time()),
    )
    print("best trial:", study.best_trial.number, round(study.best_value, 4), study.best_params)
except Exception:
    traceback.print_exc()

## Stage C: best trials, longer, folds 0 and 1, with test predictions

In [ ]:
try:
    if study is None:
        import optuna
        study = optuna.load_study(study_name="B", storage=f"sqlite:///{sweep.tracking.MLRUNS_DIR / 'optuna.db'}")
    finals = sweep.final_configs(study, top_k=2, epochs=FINAL_EPOCHS, folds=(0, 1))
    sweep.run_list(finals, RESULTS, deadline=T0 + TOTAL_HOURS * 3600)
except Exception:
    traceback.print_exc()

## Summary

In [ ]:
table = results.summarize(RESULTS)
table.to_csv(RESULTS / "summary_all.csv", index=False)
cols = [c for c in ("name", "status", "best_score", "f1_macro_05", "f1_macro_tuned", "sec_per_epoch", "num_parameters",
                    "config.lr", "config.weight_decay", "config.activation", "config.dropout", "config.pos_weight",
                    "config.use_batchnorm", "config.epochs", "config.fold") if c in table]
display(table[cols])
print(f"total time {(time.time() - T0) / 3600:.2f} h; everything is in {RESULTS}")